# M11.5：官方 Bonsai 27B artifact 能力探针

**问题**：官方三值 27B 比原版 FP 弱多少？我们的转移效率框架预测：ppl 近平、判别打折扣、自由回忆最弱。本实验直接实测（官方 artifact 首次能力探针）。

**配对协议**：同一个 C++ 打分器 + 同一份 TSV 跑双模型——
- 三值侧：`Ternary-Bonsai-2-27B-PQ2_0.gguf`（官方 artifact，7.2GB，私有 HF 仓中转）
- FP 参照：`unsloth/Qwen3.8-27B-Q8_0.gguf`（29GB，量化损失 <1%）
- 探针：去污染 sciq（845×4）+ ARC-c（1172×4），闭卷格式与我们全部 1.7B 实验一致

**运行**：A100 ~2.5h（构建 25min + 下载 35min + 探针 40min + ppl/抽检 25min）。

In [ ]:
%pip -q install cmake huggingface_hub
!test -d /content/llama.cpp || git clone -q --depth 1 -b prism https://github.com/PrismML-Eng/llama.cpp.git /content/llama.cpp
print("fork ready")

In [ ]:
# CUDA 构建（~25 min；目标：server/cli/perplexity + libllama 供打分器链接）
import subprocess, os
CUDA_ARCH = os.popen("nvidia-smi --query-gpu=compute_cap --format=csv,noheader").read().strip().replace(".", "")
print("CUDA arch:", CUDA_ARCH)
r = subprocess.run("cmake -B build-cuda -DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=ON -DGGML_CUDA=ON "
                   f"-DGGML_CUDA_ARCH={CUDA_ARCH} /content/llama.cpp > /tmp/cmake.log 2>&1", shell=True)
if r.returncode: print(open("/tmp/cmake.log").read()[-2000:])
r = subprocess.run("cmake --build build-cuda --target llama-server llama-cli llama-perplexity -j$(nproc) "
                   "> /tmp/build.log 2>&1", shell=True)
if r.returncode:
    print(open("/tmp/build.log").read()[-3000:])
else:
    print(open("/tmp/build.log").read()[-200:])

In [ ]:
# 写入并编译打分器（链接 build-cuda 的 libllama；本机已对 v07 冒烟验证）
SCORER = r"""
// probe_scorer: paired likelihood scoring for knowledge probes (M11.5)
// 用法: probe_scorer <model.gguf> <pairs.tsv> <out.tsv>
//   pairs.tsv 每行: prompt<TAB>option （\n 转义为 \\n，\t 转义为 \\t）
//   out.tsv   每行: logprob_sum<TAB>n_tok
// 算法与 PyTorch 探针一致：单次 decode(prompt+option)，
// 取位置 pl-1..end-2 的 logits，对 option token 求 log_softmax 之和。
#include "llama.h"
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <fstream>
#include <iostream>
#include <sstream>
#include <string>
#include <vector>

static std::string unescape(const std::string & s) {
    std::string r; r.reserve(s.size());
    for (size_t i = 0; i < s.size(); ++i) {
        if (s[i] == '\\' && i + 1 < s.size()) {
            if (s[i+1] == 'n') { r += '\n'; ++i; continue; }
            if (s[i+1] == 't') { r += '\t'; ++i; continue; }
            if (s[i+1] == '\\') { r += '\\'; ++i; continue; }
        }
        r += s[i];
    }
    return r;
}

int main(int argc, char ** argv) {
    if (argc < 4) {
        fprintf(stderr, "usage: %s <model.gguf> <pairs.tsv> <out.tsv>\n", argv[0]);
        return 1;
    }
    llama_backend_init();
    int ngl = argc > 4 ? atoi(argv[4]) : 0;
    auto mp = llama_model_default_params();
    mp.n_gpu_layers = ngl;
    llama_model * mi = llama_model_load_from_file(argv[1], mp);
    if (!mi) { fprintf(stderr, "model load failed\n"); return 1; }

    const int n_ctx = 1024;
    auto cp = llama_context_default_params();
    cp.n_ctx = n_ctx;
    cp.n_batch = n_ctx;
    cp.no_perf = true;
    llama_context * ctx = llama_init_from_model(mi, cp);
    if (!ctx) { fprintf(stderr, "ctx init failed\n"); return 1; }
    const struct llama_vocab * vocab = llama_model_get_vocab(mi);
    const int n_vocab = llama_vocab_n_tokens(vocab);

    std::ifstream in(argv[2]);
    std::ofstream out(argv[3]);
    std::string line;
    size_t n_done = 0;
    while (std::getline(in, line)) {
        auto tab = line.find('\t');
        if (tab == std::string::npos) continue;
        std::string prompt = unescape(line.substr(0, tab));
        std::string option = unescape(line.substr(tab + 1));

        std::vector<llama_token> tp, tall;
        tp.resize(prompt.size() + 8);
        int np = llama_tokenize(vocab, prompt.data(), prompt.size(), tp.data(), (int)tp.size(), false, false);
        if (np < 0) { tp.resize(-np); np = llama_tokenize(vocab, prompt.data(), prompt.size(), tp.data(), (int)tp.size(), false, false); }
        tp.resize(np);
        std::string full = prompt + option;
        tall.resize(full.size() + 8);
        int na = llama_tokenize(vocab, full.data(), full.size(), tall.data(), (int)tall.size(), false, false);
        if (na < 0) { tall.resize(-na); na = llama_tokenize(vocab, full.data(), full.size(), tall.data(), (int)tall.size(), false, false); }
        tall.resize(na);
        const int pl = (int)tp.size();
        if (na <= pl || na > n_ctx - 2) { out << "NA\t0\n"; continue; }

        // 单次 batch：全序列，logits 仅在 pl-1 .. na-2
        llama_batch batch = llama_batch_init(na, 0, 1);
        std::vector<bool> want(na, false);
        for (int i = pl - 1; i <= na - 2; ++i) want[i] = true;
        for (int i = 0; i < na; ++i) {
            batch.token[i] = tall[i];
            batch.pos[i] = i;
            batch.n_seq_id[i] = 1;
            batch.seq_id[i][0] = 0;
            batch.logits[i] = want[i] ? 1 : 0;
        }
        batch.n_tokens = na;
        if (llama_decode(ctx, batch) != 0) { out << "NA\t0\n"; llama_batch_free(batch); continue; }

        double lp_sum = 0.0; int n_tok = na - pl;
        for (int i = pl - 1; i <= na - 2; ++i) {
            const float * lg = llama_get_logits_ith(ctx, i);
            float mx = lg[0];
            for (int v = 1; v < n_vocab; ++v) if (lg[v] > mx) mx = lg[v];
            double z = 0.0;
            for (int v = 0; v < n_vocab; ++v) z += std::exp((double)lg[v] - mx);
            const double lse = std::log(z) + mx;
            lp_sum += (double)lg[tall[i + 1]] - lse;
        }
        out.precision(10);
        out << lp_sum << "\t" << n_tok << "\n";
        llama_batch_free(batch);
        llama_memory_clear(llama_get_memory(ctx), true);
        if (++n_done % 500 == 0) { fprintf(stderr, "  scored %zu\n", n_done); fflush(stderr); }
    }
    out.close();
    fprintf(stderr, "done: %zu pairs\n", n_done);
    llama_free(ctx); llama_model_free(mi); llama_backend_free();
    return 0;
}

"""
open("/content/probe_scorer.cpp", "w").write(SCORER)
import subprocess
import os, glob
so = glob.glob("/content/build-cuda/**/libllama.so*", recursive=True)
assert so, "libllama.so 不存在——确认上一格 cmake 带 -DBUILD_SHARED_LIBS=ON 且构建成功"
libdirs = sorted(set(os.path.dirname(x) for x in
                     glob.glob("/content/build-cuda/**/lib*.so*", recursive=True)))
L = " ".join("-L" + d for d in libdirs)
R = " ".join("-Wl,-rpath," + d for d in libdirs)
cmd = ("g++ -O2 -o /content/probe_scorer /content/probe_scorer.cpp "
       "-I/content/llama.cpp/include -I/content/llama.cpp/ggml/include "
       + L + " -lllama -lggml -lggml-base -lggml-cuda " + R)
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:] if r.stderr else "")
assert os.path.exists("/content/probe_scorer"), "scorer build failed"
print("probe_scorer ready | libdirs:", libdirs)

In [ ]:
# 下载双模型（三值 7.2GB 私有仓 + Q8 29GB）
import os
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    TOK = userdata.get("HF_TOKEN")
except Exception:
    import getpass; TOK = getpass.getpass("HF token: ")

TERN = hf_hub_download("benzeng/tmp-ternary-bonsai-27b", "Ternary-Bonsai-2-27B-PQ2_0.gguf",
                       repo_type="model", token=TOK)
print("ternary:", TERN)
FPQ8 = hf_hub_download("unsloth/Qwen3.8-27B-GGUF", "Qwen3.8-27B-Q8_0.gguf",
                       repo_type="model", token=TOK)
print("fp-q8:", FPQ8)

In [ ]:
# 构建探针 TSV（与我们 1.7B 实验完全同格式；\n → \\n 转义）
import os
os.environ["HF_HUB_OFFLINE"] = "0"
from datasets import load_dataset

def ngrams(text, n=13):
    ws = text.lower().split()
    return {" ".join(ws[i:i+n]) for i in range(len(ws)-n+1)} if len(ws)>=n else set()

_train_ng = set()
for ex in load_dataset("allenai/sciq", split="train"):
    _train_ng |= ngrams(ex["support"] + " " + ex["question"])
sciq_val_all = load_dataset("allenai/sciq", split="validation")
CLEAN_IDX = [i for i, ex in enumerate(sciq_val_all)
             if not (ngrams(ex["support"] + " " + ex["question"]) & _train_ng)]
sciq_val = sciq_val_all.select(CLEAN_IDX)
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
print(f"sciq clean {len(sciq_val)}/1000 | ARC {len(arc)}")

def esc(s): return s.replace("\\", "\\\\").replace("\n", "\\n").replace("\t", "\\t")
pairs = []   # (probe_tag, ex_idx, opt_idx, prompt, option)
for qi, ex in enumerate(sciq_val):
    prompt = f"Question: {ex['question']}\nAnswer:" + "\n"
    opts = [ex["correct_answer"]] + [ex[f"distractor{i}"] for i in (1, 2, 3)]
    for oi, o in enumerate(opts):
        pairs.append(("sciq", qi, oi, prompt, " " + o))
for qi, ex in enumerate(arc):
    prompt = f"Question: {ex['question']}\nAnswer:" + "\n"
    for oi, o in enumerate(ex["choices"]["text"]):
        pairs.append(("arc", qi, oi, prompt, " " + o))
with open("/content/pairs.tsv", "w") as f:
    for tag, qi, oi, p, o in pairs:
        f.write(f"{esc(p)}\t{esc(o)}\n")
with open("/content/pairs.meta", "w") as f:
    for tag, qi, oi, p, o in pairs:
        f.write(f"{tag}\t{qi}\t{oi}\n")
print(f"pairs: {len(pairs)}")

In [ ]:
# 双模型跑探针（GPU 全卸载；每对一次 decode）
import subprocess, time, glob
TERN = glob.glob("/root/.cache/huggingface/hub/models--benzeng--tmp-ternary-bonsai-27b/snapshots/*/*.gguf")[0]
FPQ8 = glob.glob("/root/.cache/huggingface/hub/models--unsloth--Qwen3.8-27B-GGUF/snapshots/*/Qwen3.8-27B-Q8_0.gguf")[0]

for name, path in (("ternary-27B", TERN), ("fp-q8-27B", FPQ8)):
    t0 = time.time()
    r = subprocess.run(["/content/probe_scorer", path, "/content/pairs.tsv",
                        f"/content/scores_{name}.tsv", "99"],
                       capture_output=True, text=True)
    print(f"[{name}] {time.time()-t0:.0f}s {r.stderr.strip().splitlines()[-1]}")

In [ ]:
# 判卷：配对计算 acc / acc_norm + 框架对照表
import math, os
meta = [l.split("\t") for l in open("/content/pairs.meta")]
meta = [(t, int(q), int(o)) for t, q, o in meta]

def read_scores(name):
    vals = []
    for line in open(f"/content/scores_{name}.tsv"):
        a, b = line.strip().split("\t")
        vals.append((float(a), int(b)))
    return vals

results = {}
for name in ("ternary-27B", "fp-q8-27B"):
    sc = read_scores(name)
    by = {}
    for (tag, qi, oi), (lp, n) in zip(meta, sc):
        by.setdefault((tag, qi), {})[oi] = (lp, n)
    for tag in ("sciq", "arc"):
        gold = 0 if tag == "sciq" else None
        acc = accn = n = 0
        for (t, qi), opts in by.items():
            if t != tag: continue
            if tag == "arc":
                # gold = 正确选项序（choices 顺序里 answerKey 的位置）——由 pairs 构建可知 sciq gold=0；
                # ARC 的 gold 需要重放 answerKey
                continue
            best_a = max(opts, key=lambda i: opts[i][0])
            best_n = max(opts, key=lambda i: opts[i][0]/max(opts[i][1],1))
            acc += best_a == 0; accn += best_n == 0; n += 1
        if tag == "sciq":
            results[(name, "sciq")] = (acc/n, accn/n, n)
            print(f"[{name} sciq] acc {acc/n:.3f} | acc_norm {accn/n:.3f} (n={n})")

# ARC gold 重放（answerKey → 选项序）
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["HF_DATASETS_OFFLINE"] = "1"
from datasets import load_dataset
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
gold_arc = {i: ex["choices"]["label"].index(ex["answerKey"]) for i, ex in enumerate(arc)}
import os
for name in ("ternary-27B", "fp-q8-27B"):
    sc = read_scores(name)
    by = {}
    for (tag, qi, oi), (lp, n) in zip(meta, sc):
        if tag == "arc":
            by.setdefault(qi, {})[oi] = (lp, n)
    acc = accn = n = 0
    for qi, opts in by.items():
        g = gold_arc[qi]
        best_a = max(opts, key=lambda i: opts[i][0])
        best_n = max(opts, key=lambda i: opts[i][0]/max(opts[i][1],1))
        acc += best_a == g; accn += best_n == g; n += 1
    results[(name, "arc")] = (acc/n, accn/n, n)
    print(f"[{name} ARC] acc {acc/n:.3f} | acc_norm {accn/n:.3f} (n={n})")

print()
print("=== 框架对照（acc_norm）===")
rows = [
    ("官方 27B 三值", results.get(("ternary-27B", "sciq"), (float('nan'),)*3)[1], results.get(("ternary-27B", "arc"), (float('nan'),)*3)[1]),
    ("27B Q8 参照",   results.get(("fp-q8-27B", "sciq"), (float('nan'),)*3)[1], results.get(("fp-q8-27B", "arc"), (float('nan'),)*3)[1]),
    ("我方 v0.7 1.7B", 0.530, 0.319),
    ("1.7B FP", 0.699, 0.378),
    ("8B 教师(PyTorch)", 0.830, 0.472),
]
print("| 模型 | sciq | ARC |")
print("|---|---|---|")
for r in rows: print(f"| {r[0]} | {r[1]:.3f} | {r[2]:.3f} |")
t = results.get(("ternary-27B", "sciq"), (float('nan'),)*3)[1]
f = results.get(("fp-q8-27B", "sciq"), (float('nan'),)*3)[1]
if f and f == f:
    print(f"\n官方三值/FP 参照（sciq acc_norm）= {t/f:.1%}；对照我方 1.7B 框架：v0.7/8B教师 = {0.530/0.830:.1%}")

In [ ]:
# 抽检：三题双模型（chat 模板 + 裸续写，强制配方）
import glob, subprocess, time
BIN = "/content/build-cuda/bin/llama-cli"
TERN = glob.glob("/root/.cache/huggingface/hub/models--benzeng--tmp-ternary-bonsai-27b/snapshots/*/*.gguf")[0]
FPQ8 = glob.glob("/root/.cache/huggingface/hub/models--unsloth--Qwen3.8-27B-GGUF/snapshots/*/Qwen3.8-27B-Q8_0.gguf")[0]
QS = ["What planet is known as the Red Planet?",
      "Why do we see lightning before we hear thunder?",
      "What is the powerhouse of the cell?"]
for name, path in (("ternary-27B", TERN), ("fp-q8-27B", FPQ8)):
    for q in QS:
        r = subprocess.run([BIN, "-m", path, "-p", q, "-n", "48", "-t", "8", "-st", "--simple-io",
                            "--temp", "0.5", "--top-p", "0.85", "--top-k", "20",
                            "--repeat-penalty", "1.1", "-ngl", "99"],
                           capture_output=True, text=True, timeout=300)
        txt = r.stdout.split(q)[-1].strip()[:200]
        print(f"[{name}] Q: {q}\nA(raw): {txt}\n", flush=True)

In [ ]:
# ppl 交叉验证（wt2_30k c512，官方白皮书宣称 1.02×FP）
import glob, subprocess
TERN = glob.glob("/root/.cache/huggingface/hub/models--benzeng--tmp-ternary-bonsai-27b/snapshots/*/*.gguf")[0]
FPQ8 = glob.glob("/root/.cache/huggingface/hub/models--unsloth--Qwen3.8-27B-GGUF/snapshots/*/Qwen3.8-27B-Q8_0.gguf")[0]
from datasets import load_dataset
text2 = "\n\n".join(t for t in load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")["test"]["text"] if t.strip())
open("/content/wt2_30k.txt", "w").write(text2[:120000])
for name, path in (("ternary-27B", TERN), ("fp-q8-27B", FPQ8)):
    r = subprocess.run(["/content/build-cuda/bin/llama-perplexity", "-m", path,
                        "-f", "/content/wt2_30k.txt", "-c", "512", "-t", "8", "-ngl", "99"],
                       capture_output=True, text=True, timeout=1800)
    for line in r.stderr.splitlines():
        if "Final estimate" in line or "PPL =" in line:
            print(f"[{name}] {line.strip()}")